# Real-Time Monitoring of Oil and Gas Assets Using Edge AI
### Classical Machine Learning Pipeline for LPG Transmission Pipeline Fault Detection

**Objective**: Replace deep-learning architectures (CNN-LSTM) with efficient, physics-informed classical machine learning models (Scikit-Learn) tailored for real-time edge execution (e.g., ESP32 / industrial RTUs).

**Key Architecture Highlights**:
1. **Leakage-Free 60-Second Windows**: Window length = $60\text{ s} / 5\text{ s} = 12$ samples. Sensor inputs strictly limited to 6 physical channels (`pressure_bar_g`, `flow_mmscmd`, `gas_temp_c`, `vibration_mm_s_rms`, `ultrasonic_db_rel`, `gas_lel_pct`).
2. **Physics-Informed Flow Residuals**: Empirical regression of pressure, vibration, and acoustic emissions as functions of flow on *healthy training data only* ($P - \hat{P}(Q)$).
3. **Two-Stage Detection & Prediction**:
   - **Unsupervised Anomaly Detectors**: Isolation Forest, One-Class SVM, and Mahalanobis Distance calibrated to the 99.9th percentile of healthy validation data.
   - **Supervised Classifiers**: HistGradientBoosting & Random Forest distinguishing Normal, Leak, and Blockage (with demand steps and sensor faults mapped to Normal).
   - **Calibrated Danger Predictor**: 10-minute lookahead warning calibrated to a false-alarm budget of $\le 2$ alarms/day.
4. **Full Multi-Seed, Ablation, and Edge Latency Verification**.

In [ ]:
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project directory is in python path
sys.path.append(os.path.abspath('lpg_pipeline_monitor'))

from ml_pipeline import (
    prepare_dataset, SENSOR_COLS, CLASS_NAMES,
    FlowResidualRegressor, MahalanobisDetector,
    WINDOW_LEN, SAMPLES_PER_DAY_PER_NODE
)
from predict_service import predict_window

np.random.seed(42)
print("Libraries successfully loaded. Ready for ML pipeline execution.")

## 1. Data Ingestion, Leakage Prevention & Flow-Compensated Residuals

Sensor dropouts are handled per-node via forward then backward fill. All bookkeeping labels (`limit_alarm`, `label`, `fault_type`, `event_id`, etc.) are stripped from model inputs to prevent target leakage.

In [ ]:
csv_path = 'data_training/pipeline_training_data.csv' if os.path.exists('data_training/pipeline_training_data.csv') else 'pipeline_training_data.csv'
dataset = prepare_dataset(csv_path)

meta = dataset['df_meta']
X = dataset['X']
y_fault = dataset['y_fault']
y_danger = dataset['y_danger']
res_reg = dataset['res_reg']
feature_names = dataset['feature_names']

print(f"Extracted {X.shape[1]} features across {len(X):,} valid 60s windows.")
print(f"Flow Residual Model Slopes:")
print(f"  Pressure vs Flow:   {res_reg.reg_p.coef_[0]:.4f} bar/MMSCMD (Intercept: {res_reg.reg_p.intercept_:.2f})")
print(f"  Vibration vs Flow:  {res_reg.reg_v.coef_[0]:.4f} mm/s/MMSCMD (Intercept: {res_reg.reg_v.intercept_:.2f})")
print(f"  Ultrasonic vs Flow: {res_reg.reg_u.coef_[0]:.4f} dB/MMSCMD (Intercept: {res_reg.reg_u.intercept_:.2f})")

## 2. Train / Val / Test Partitioning

Strict temporal split adherence (no cross-split shuffling). Healthy validation data is reserved for threshold calibration.

In [ ]:
train_mask = meta['split'] == 'train'
val_mask = meta['split'] == 'val'
test_mask = meta['split'] == 'test'

healthy_train_mask = train_mask & (meta['fault_type'] == 'none') & (meta['event_kind'].isna())
healthy_val_mask = val_mask & (meta['fault_type'] == 'none') & (meta['event_kind'].isna())

X_train, y_train = X[train_mask], y_fault[train_mask]
X_val, y_val = X[val_mask], y_fault[val_mask]
X_test, y_test = X[test_mask], y_fault[test_mask]

print(f"Train samples: {len(X_train):,} (Healthy: {healthy_train_mask.sum():,})")
print(f"Val samples:   {len(X_val):,} (Healthy: {healthy_val_mask.sum():,})")
print(f"Test samples:  {len(X_test):,}")

## 3. Supervised Fault Classification (Normal / Leak / Blockage)

Comparing HistGradientBoosting and Random Forest against baseline SCADA static limits.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# HistGradientBoosting
hgb = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
hgb.fit(X_train, y_train)
hgb_preds = hgb.predict(X_test)

print("=== HistGradientBoosting Test Report ===")
print(classification_report(y_test, hgb_preds, target_names=CLASS_NAMES, digits=4))
print(f"Accuracy: {accuracy_score(y_test, hgb_preds):.4f} | Macro-F1: {f1_score(y_test, hgb_preds, average='macro'):.4f}")

## 4. Calibrated Danger Predictor (10-Minute Breach Lookahead)

Predicts whether static limits will trip within the next 10 minutes. Calibrated via sigmoid scaling, tuned on validation to satisfy the $\le 2$ false-alarms/day budget.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

base_danger = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
cal_danger = CalibratedClassifierCV(estimator=base_danger, method='sigmoid', cv=3)
cal_danger.fit(X_train, y_danger[train_mask])

test_danger_probs = cal_danger.predict_proba(X_test)[:, 1]
brier = brier_score_loss(y_danger[test_mask], test_danger_probs)
print(f"Danger Predictor Brier Score: {brier:.4f}")

# Reliability Curve
prob_true, prob_pred = calibration_curve(y_danger[test_mask], test_danger_probs, n_bins=10)
plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Calibrated Model')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Fraction of Positives')
plt.title(f'Reliability Curve (Brier Score = {brier:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Ablation Study & Multi-Seed Verification (5 Runs)

Inspect feature set contributions and test model stability across 5 independent seeds.

In [ ]:
ablation_df = pd.read_csv('results/ablation_study.csv')
print("=== ABLATION STUDY RESULTS ===")
display(ablation_df)

seed_df = pd.read_csv('results/multi_seed_metrics.csv')
print("\n=== MULTI-SEED METRICS (5 SEEDS) ===")
display(seed_df)
print(f"HGB Macro-F1: {seed_df['hgb_macro_f1'].mean():.4f} +/- {seed_df['hgb_macro_f1'].std():.4f}")
print(f"HGB Accuracy: {seed_df['hgb_accuracy'].mean():.4f} +/- {seed_df['hgb_accuracy'].std():.4f}")

## 6. Model Explainability: Tree SHAP & Permutation Importance

Displays the feature drivers isolating leaks and blockages.

In [ ]:
if os.path.exists('results/shap_summary.png'):
    from IPython.display import Image, display
    display(Image(filename='results/shap_summary.png'))
else:
    print("SHAP summary image not found in results/")

## 7. Edge Inference Testing: `predict_window(df_window)`

Benchmarking latency and verifying real-time edge outputs on individual 60s telemetry windows.

In [ ]:
test_df = meta[test_mask]

# Sample a mature leak window
leak_window = test_df[(test_df['fault_type'] == 'leak') & (test_df['s_since_onset'] > 300)].iloc[:12]
out = predict_window(leak_window, models_dir='lpg_pipeline_monitor/models')

print("Live Window Prediction:")
for k, v in out.items():
    print(f"  {k:20s}: {v}")